In [2]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\avina\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import torch
torch.__version__

'2.6.0+cu124'

In [4]:
torch.cuda.is_available()

True

In [5]:
# creating pyTorch tensor
import torch
tensor0d=torch.tensor(4)
tensor1d=torch.tensor([1,2,3])
tensor2d=torch.tensor([[1,2,3],[4,5,6]])


In [6]:
tensor1d.dtype

torch.int64

In [7]:
float_tensor=torch.tensor([1.0,2.0,3.0])
float_tensor.dtype

torch.float32

In [8]:
float_tensor=tensor1d.type(torch.float32)
float_tensor.dtype

torch.float32

In [9]:
# common tensor operations
print(tensor2d.shape)

torch.Size([2, 3])


In [10]:
print(tensor2d.reshape(3,2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [11]:
print(tensor2d.view(3,2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [12]:
# transpose not same as reshape or view
print(tensor2d.T)

tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [13]:
# tensor multiplication 
print(tensor2d @ tensor2d.T)

tensor([[14, 32],
        [32, 77]])


In [14]:
import torch.nn.functional as F  # functional API for loss functions

# target label (ground truth)
y = torch.tensor([1.0])

# input feature and model parameters
x1 = torch.tensor([1.1])       # input value
w1 = torch.tensor([2.2])       # weight
b = torch.tensor([0.0])        # bias

# linear combination: z = w*x + b
z = w1 * x1 + b

# apply sigmoid to get probability
a = torch.sigmoid(z)

# binary cross-entropy loss between prediction and target
loss = F.binary_cross_entropy(a, y)

# print final loss value
print(loss)

tensor(0.0852)


In [15]:
# computing gradients via autograd
import torch.nn.functional as F
from torch.autograd import grad

y= torch.tensor([1.0])
x1=torch.tensor([1.1])
w1=torch.tensor([2.2], requires_grad=True)
b=torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

# compute gradients of loss w.r.t. parameters
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [16]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [17]:
# a multilayer perceptron with two hidden layers
class NeuralNetwork(torch.nn.Module):
    def __init__(self,num_inputs,num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # input layer to hidden layer 1
            torch.nn.Linear(num_inputs,30),
            torch.nn.ReLU(),
            # hidden layer 1 to hidden layer 2
            torch.nn.Linear(30,20),
            torch.nn.ReLU(),
            # hidden layer 2 to output layer
            torch.nn.Linear(20,num_outputs)
        )
    def forward(self,x):
        logits = self.layers(x)
        return logits
    

In [18]:
model=NeuralNetwork(num_inputs=50,num_outputs=3)

In [19]:
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [20]:
# total number of parameters
num_params=sum(p.numel() for p in model.parameters() if p.requires_grad)

In [21]:
num_params

2213

In [22]:
model.layers[0].weight

Parameter containing:
tensor([[-0.0736, -0.1229,  0.0392,  ..., -0.1345, -0.0061, -0.1379],
        [ 0.1067, -0.1358, -0.0117,  ..., -0.0686, -0.0031,  0.0418],
        [-0.0158,  0.0986, -0.1258,  ...,  0.0861, -0.1234,  0.0566],
        ...,
        [-0.0731, -0.0388,  0.0873,  ...,  0.1158, -0.0331, -0.0587],
        [ 0.0191, -0.1305,  0.0098,  ...,  0.0136, -0.0718, -0.1346],
        [-0.0648,  0.0381, -0.1213,  ...,  0.0582, -0.0751,  0.0111]],
       requires_grad=True)

In [23]:
model.layers[0].weight.shape

torch.Size([30, 50])

In [24]:
# reproduceable weights
torch.manual_seed(123)
model=NeuralNetwork(num_inputs=50,num_outputs=3)


In [25]:
model.layers[0].weight

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)

In [26]:
# inspecting forward pass
torch.manual_seed(123)
X=torch.rand((1,50))



In [27]:
X

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274,
         0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584, 0.2775,
         0.8573, 0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
         0.0772, 0.3565, 0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606,
         0.5159, 0.4220, 0.5786, 0.9455, 0.8057]])

In [28]:
out=model(X)

In [29]:
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [30]:
# if we are using models for prediction only we can turn off gradients to save memory and computations
with torch.no_grad():
    out=model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [31]:
# using softmax to get probabilities from logits
with torch.no_grad():
    out=model(X)
    probs=F.softmax(out,dim=1)
probs

tensor([[0.3113, 0.3934, 0.2952]])

In [32]:
# Setting up efficient data loaders
X_train=torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train=torch.tensor([0,0,0,1,1])
X_test=torch.tensor([
    [-0.8, 2.8],
    [2.6,-1.6]
])

y_test=torch.tensor([0,1])

In [42]:
#defining a dataset class
from torch.utils.data import Dataset
class ToyDataset(Dataset):
    def __init__(self,X,y):
        self.features=X
        self.labels=y
    def __getitem__(self,idx):
        one_x=self.features[idx]
        one_y=self.labels[idx]
        return one_x,one_y
    def __len__(self):
        return self.labels.shape[0]


In [43]:
train_ds=ToyDataset(X_train,y_train)
test_ds=ToyDataset(X_test,y_test)


In [44]:
len(train_ds)

5

In [45]:
# using dataloader
from torch.utils.data import DataLoader
torch.manual_seed(123)

train_loader=DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

test_loader=DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)


In [ ]:
# iterating through the dataloader
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


 In practice, having a substantially smaller batch as the last batch in a training epoch can disturb the convergence during training. To prevent this, set drop_last=True

In [ ]:
train_loader=DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)


In [ ]:
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


In [46]:
# training our neural network
import torch.nn.functional as F

torch.manual_seed(123)
model=NeuralNetwork(num_inputs=2,num_outputs=2)
optimizer=torch.optim.SGD(model.parameters(),lr=0.5)

num_epochs=3
for epoch in range(num_epochs):
    model.train()
    for batch_idx,(features,labels) in enumerate(train_loader):
        logits=model(features)
        loss=F.cross_entropy(logits,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # logging
        print(f"Epoch {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Loss: {loss.item():.2f}")
# evaluating our model
model.eval()
with torch.no_grad():
    outputs=model(X_train)
print("Train outputs:", outputs)
    

Epoch 001/003 | Batch 001/003 | Loss: 0.75
Epoch 001/003 | Batch 002/003 | Loss: 0.65
Epoch 001/003 | Batch 003/003 | Loss: 0.42
Epoch 002/003 | Batch 001/003 | Loss: 0.05
Epoch 002/003 | Batch 002/003 | Loss: 0.13
Epoch 002/003 | Batch 003/003 | Loss: 0.00
Epoch 003/003 | Batch 001/003 | Loss: 0.01
Epoch 003/003 | Batch 002/003 | Loss: 0.00
Epoch 003/003 | Batch 003/003 | Loss: 0.02
Train outputs: tensor([[ 2.9320, -4.2563],
        [ 2.6045, -3.8389],
        [ 2.1484, -3.2514],
        [-2.1461,  2.1496],
        [-2.5004,  2.5210]])


In [47]:
# obtaining class memberships using softmax
torch.set_printoptions(sci_mode=False)
probas=torch.softmax(outputs,dim=1)
print(probas)

tensor([[    0.9992,     0.0008],
        [    0.9984,     0.0016],
        [    0.9955,     0.0045],
        [    0.0134,     0.9866],
        [    0.0066,     0.9934]])


In [51]:
# printing predicted class labels
pred_labels=torch.argmax(probas,dim=1)
print(pred_labels)

tensor([0, 0, 0, 1, 1])


In [48]:
# we can also get predicted class labels directly from the logits using argmax
pred_labels_direct=torch.argmax(outputs,dim=1)
print(pred_labels_direct)

tensor([0, 0, 0, 1, 1])


In [52]:
torch.sum(pred_labels==y_train)

tensor(5)

In [53]:
def accuracy(model, dataloader):
    model=model.eval()
    correct=0
    total_examples=0

    for idx, (features,labels) in enumerate(dataloader):
        with torch.no_grad():
            logits=model(features)
        predictions=torch.argmax(logits,dim=1)
        compare=predictions==labels
        correct+=torch.sum(compare)
        total_examples+=len(compare)
    acc=correct/total_examples
    return acc.item()

In [54]:
accuracy(model, train_loader)

1.0

In [55]:
# saving model
torch.save(model.state_dict(),"model.pth")

In [57]:
# restoring model
model2=NeuralNetwork(num_inputs=2,num_outputs=2)
model2.load_state_dict(torch.load("model.pth"))

<All keys matched successfully>

In [58]:
model2.eval()

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)

In [59]:
torch.cuda.is_available()

True

In [61]:
tensor1=torch.tensor([1.0,2.0,3.0])
tensor2=torch.tensor([4.0,5.0,6.0])
tensor_sum=tensor1+tensor2
print(tensor_sum)

tensor([5., 7., 9.])


In [62]:
# using GPU for tensor computations
tensor1=tensor1.to("cuda")
tensor2=tensor2.to("cuda")
print(tensor1 + tensor2)


tensor([5., 7., 9.], device='cuda:0')


In [63]:
tensor1=tensor1.to("cpu")
print(tensor1+tensor2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [64]:
# training on GPU
torch.manual_seed(123)
model=NeuralNetwork(num_inputs=2,num_outputs=2)
device= "cuda" if torch.cuda.is_available() else "cpu"
model=model.to(device)

optimizer=torch.optim.SGD(model.parameters(),lr=0.5)

num_epochs=3

for epoch in range(num_epochs):
    model.train()
    for batch_idx,(features,labels) in enumerate(train_loader):
        features,labels=features.to(device),labels.to(device)
        logits=model(features)
        loss=F.cross_entropy(logits,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Loss: {loss.item():.2f}")
    model.eval()

Epoch 001/003 | Batch 001/003 | Loss: 0.75
Epoch 001/003 | Batch 002/003 | Loss: 0.65
Epoch 001/003 | Batch 003/003 | Loss: 0.42
Epoch 002/003 | Batch 001/003 | Loss: 0.05
Epoch 002/003 | Batch 002/003 | Loss: 0.13
Epoch 002/003 | Batch 003/003 | Loss: 0.00
Epoch 003/003 | Batch 001/003 | Loss: 0.01
Epoch 003/003 | Batch 002/003 | Loss: 0.00
Epoch 003/003 | Batch 003/003 | Loss: 0.02


In [65]:
# Compare CPU vs GPU matrix multiplication performance
import time

# Test different matrix sizes
sizes = [100, 500, 1000, 2000, 5000]
results = []

device = "cuda" if torch.cuda.is_available() else "cpu"

for size in sizes:
    # Create random matrices
    a_cpu = torch.randn(size, size)
    b_cpu = torch.randn(size, size)
    
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")
    
    # Warm up
    _ = a_cpu @ b_cpu
    if torch.cuda.is_available():
        _ = a_gpu @ b_gpu
        torch.cuda.synchronize()
    
    # Time CPU
    cpu_time = %timeit -o a_cpu @ b_cpu
    
    # Time GPU
    if torch.cuda.is_available():
        gpu_time = %timeit -o a_gpu @ b_gpu
        speedup = cpu_time.best / gpu_time.best
        results.append({
            "size": size,
            "cpu_time": cpu_time.best * 1000,  # convert to ms
            "gpu_time": gpu_time.best * 1000,
            "speedup": speedup
        })
        print(f"Matrix size: {size}x{size}")
        print(f"  CPU time: {cpu_time.best * 1000:.4f} ms")
        print(f"  GPU time: {gpu_time.best * 1000:.4f} ms")
        print(f"  Speedup: {speedup:.2f}x\n")
    else:
        print(f"Matrix size: {size}x{size}")
        print(f"  CPU time: {cpu_time.best * 1000:.4f} ms")
        print(f"  GPU not available\n")

16.1 µs ± 595 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
26.5 µs ± 547 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Matrix size: 100x100
  CPU time: 0.0154 ms
  GPU time: 0.0257 ms
  Speedup: 0.60x

767 µs ± 13.3 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
91.3 µs ± 4.75 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Matrix size: 500x500
  CPU time: 0.7500 ms
  GPU time: 0.0843 ms
  Speedup: 8.90x

5.53 ms ± 204 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
565 µs ± 6.28 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Matrix size: 1000x1000
  CPU time: 5.2113 ms
  GPU time: 0.5600 ms
  Speedup: 9.31x

38.5 ms ± 2.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
4.23 ms ± 35.9 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Matrix size: 2000x2000
  CPU time: 33.7751 ms
  GPU time: 4.1551 ms
  Speedup: 8.13x

566 ms ± 30.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


KeyboardInterrupt: 

In [66]:
# pytorch utilities for distributed training
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
